Загружаем утилиты для быстрой подгрузки моделек

In [1]:
from utils import gemini_model_list, get_llm

Смотрим на список доступных моделей

In [2]:
gemini_model_list

['gemini-3.1-flash-lite-preview',
 'gemini-3-flash-preview',
 'gemini-2.5-pro',
 'gemini-2.5-flash',
 'gemini-2.5-flash-lite',
 'gemini-2.5-flash-lite-preview-09-2025']

Загружаем модель (рекомендуется gemini-3 и gemini-3.1)

In [ ]:
gemini = get_llm('gemini-3-flash-preview')

### Генерация введения

Далее мы будем решать следующую задачу: имея на руках понятие (концепт), сгенерировать начало статьи. С этим прекрасно справляется данный промпт:

In [16]:
index_prompt_template = """\
Напиши введение и содержание для статьи по физике для учащихся 8 класса по заданному понятию - <CONCEPT>{concept}</CONCEPT>.

## СТРУКТУРА

### # [Заголовок]
Четкое название термина, без лишних слов.

### ## Введение

1. Увлекающее начало: нужно максимально нагнать интриги крайне коротким предложением (не больше 20 слов), которое касается повседневного опыта читателя и подводит к теме. Можно начинать со слов "почему", "зачем", "когда" и так далее, за использование клише "задумывались ли вы" и "знаете ли вы" ты получаешь штраф $2000

2. Короткое определение: 2-3 предложения максимально простым языком, без сложных терминов (они будут введены позже). Формат: "Понятие - это ..."

3. Краткий обзор понятия:
   - Почему это важная характеристика
   - В каких формах/качествах проявляется (если применимо)
   - С чем не стоит путать (но только, если есть риск смешения понятий: магнитная и электромагнитная индукция, гравитация и гравитационное ускорение)

### ## Содержание

Составь оглавление статьи. Разделы должны логически и последовательно раскрывать тему: от простого к сложному, от наблюдаемого к теоретическому. Количество разделов должно раскрывать тему. Большие разделы следует разбивать на подразделы, особенно в случае перечислений.

Формат (только оглавление, без пояснений, приведён пример оглавления):
1. [Название раздела 1](#название-раздела-1)
2. [Название раздела 2](#название-раздела-2)
   a. [Название подраздела](#название-подраздела)
   b. [Название подраздела](#название-подраздела)
3. [Название раздела 3](#название-раздела-3)
... и так далее

Важно: никаких самих разделов, только введение и содержание. Текст должен быть готов к тому, что разделы будут написаны отдельно.
Ссылка на раздел оформляется так: из заголовка раздела удаляются знаки препинания, буквы приводятся к нижнему регистру, пробелы заменяются на дефисы.

<CONCEPT>{concept}</CONCEPT>
"""

Пример кода:

In [17]:
index_prompt = index_prompt_template.format(concept='Вселенная')
article = gemini.invoke(index_prompt).content[0]['text']
with open('_tmp.md', mode='w', encoding='utf-8') as file:
    file.write(article)

### Генерация наполнения статьи

Теперь необходимо распарсить ответ предыдущей модели и выделить из неё содержание:

In [43]:
index = """\
## Содержание

1. [Что такое Вселенная](#что-такое-вселенная)
2. [Масштабы и расстояния в космосе](#масштабы-и-расстояния-в-космосе)
   a. [Астрономическая единица](#астрономическая-единица)
   b. [Световой год и парсек](#световой-год-и-парсек)
3. [Структура Вселенной](#структура-вселенной)
   a. [Звезды и планетные системы](#звезды-и-планетные-системы)
   b. [Галактики и их классификация](#галактики-и-их-классификация)
   c. [Сверхскопления и крупномасштабная структура](#сверхскопления-и-крупномасштабная-структура)
4. [История развития мира](#история-развития-мира)
   a. [Теория Большого взрыва](#теория-большого-взрыва)
   b. [Расширение Вселенной и закон Хаббла](#расширение-вселенной-и-закон-хаббла)
5. [Невидимые составляющие Вселенной](#невидимые-составляющие-вселенной)
   a. [Темная материя](#темная-материя)
   b. [Темная энергия](#темная-энергия)
6. [Место Земли во Вселенной](#место-земли-во-вселенной)
7. [Будущее Вселенной](#будущее-вселенной)
"""

и названия статей:

In [44]:
parts = [
    '1. Что такое Вселенная',
    '2. Масштабы и расстояния в космосе',
    'a. Астрономическая единица',
    'b. Световой год и парсек',
    '3. Структура Вселенной',
    'a. Звезды и планетные системы',
    'b. Галактики и их классификация',
    'c. Сверхскопления и крупномасштабная структура',
    '4. История развития мира',
    'a. Теория Большого взрыва',
    'b. Расширение Вселенной и закон Хаббла',
    '5. Невидимые составляющие Вселенной',
    'a. Темная материя',
    'b. Темная энергия',
    '6. Место Земли во Вселенной',
    '7. Будущее Вселенной',
]

Причём если этого не получается сделать из-за того, что модель содержание сгенерировала неправильно, мы должны перегенерировать предыдущий шаг.

Далее мы должны пройтись по каждому пункту и сгенерировать часть статьи по нему. Вот промпт, который хорошо с этим справляется и пример кода

In [40]:
part_prompt_template = """\
Напиши раздел/подраздел <PART>{part}</PART> для статьи по физике для учащихся 8 класса.

Это часть большой статьи по термину <CONCEPT>{concept}</CONCEPT>. Пиши так, чтобы раздел органично вписывался в общий контекст.

### Содержание статьи:
{index}

Раздел - это заголовок, начинающийся с числа,
Подраздел - это заголовок, начинающийся с латинской буквы.

Твоя задача - написать раздел/подраздел - <PART>{part}</PART>

### ФОРМАТ ОТВЕТА (ОБЯЗАТЕЛЬНО):
- Ровно ОДИН заголовок уровня ### или ####
- После заголовка — ровно ОДИН абзац текста
- НИКАКИХ других заголовков (####, ### и т.д.)
- НИКАКИХ списков, подпунктов, буквенных или числовых структур

Если в ответе появится второй заголовок — ответ считается неверным.

### Требования к разделу:
- Начало файла - заголовок, начинающийся с "###", если это раздел, или с "####", если это подраздел. Далее название заголовка, без номера (числа или буквы) заголовка.
- Если в содержании у текущего раздела есть подпункты (a., b., и т.д.),
  ТО: 1. Игнорируй их содержание, 2. Не раскрывай их, 3. Напиши ТОЛЬКО вводный абзац. Это правило имеет ПРИОРИТЕТ над всеми остальными.
- Если это подраздел или раздел без подразделов, то раскрывай тему полностью, с разных сторон, но не забегая на темы следующих разделов.
- Если это первый раздел после введения, начинай с развития темы, заявленной во введении.
- Каждый раздел (и последний подраздел в рамках одного раздела) должен иметь логичное завершение.

## ТРЕБОВАНИЯ К РАЗДЕЛАМ
1. Последовательность: каждый следующий раздел опирается на понятия из предыдущих. Новые термины вводятся постепенно и сразу объясняются.
2. Логика раскрытия темы: от простого к сложному, от наблюдаемого к теоретическому, от макро- к микроуровню.
3. Хронология (если уместно): можно показать, как менялось понимание термина в истории науки.
4. Умеренные связки между разделами: статья должна читаться как единый текст, а не набор кусков, это обеспечивается вводными словами и словами-связками (Итак / Следовательно / Подводя итог), призывом (Рассмотрим что-то / Задумайтесь: / Тепер давайте перейдём). Не ограничивайся этим списком.
5. Примеры: если материал слишком абстрактный (например, магнетизм, электричество, квантовая физика), можно привести уместный и конкретный пример, иллюстрирующий материал.
6. Термины: если вводишь новый термин (например, "инертность"), выделяй его жирным и тут же давай пояснение.

### Стиль:
- Научно-популярный, для 8 класса
- Живой язык, метафоры, сравнения
- Без сложных формул
- Без воды и повторов
"""

In [ ]:
with open('_part_tmp.md', 'w') as file: file.write('')

gemini = get_llm('gemini-3.1-flash-lite-preview')

for part in parts[:4]:
    part_prompt = part_prompt_template.format(
        concept='Вселенная',
        part=part,
        index=index,
    )
    part_article = gemini.invoke(part_prompt).content[0]['text']

    with open('_part_tmp.md', mode='a', encoding='utf-8') as file:
        file.write('\n\n' + part_article)
    print('=' * 20)
    print(part)
    print('=' * 20)
    print(part_article)
    print('\n' * 3)

1. Что такое Вселенная
### 1. Что такое Вселенная

Вселенная — это не просто необъятное пространство над нашими головами, а весь существующий материальный мир, включающий в себя все небесные тела, энергию, время и даже само пространство, в котором мы находимся. В широком смысле под этим термином понимают всё, что когда-либо было, есть или будет — от мельчайших частиц материи до гигантских скоплений звезд. На протяжении веков человечество прошло путь от представлений о Земле как о центре мироздания до осознания того, что наш мир — лишь крошечная часть невероятно сложной, динамичной и постоянно расширяющейся системы, физические законы которой действуют одинаково в самых удаленных её уголках. Изучение Вселенной требует от нас способности мыслить категориями огромных масштабов, ведь то, что мы видим в телескопы, — это лишь малая доля того, что скрыто в глубинах космоса, ожидая своего открытия. Теперь, когда мы определили границы нашего интереса, давайте перейдем к тому, как ученые измеряют